#DAY 10 Databricks Challenge

##Challenges
### 🛠️ Tasks:

1. Analyze query plans
2. Partition large tables
3. Apply ZORDER
4. Benchmark improvements

####Task 1 - Analyze query plans

We will see 3 sections in the output

1. Parsed / Analyzed Logical Plan:
Confirms columns and table references,
Verifies query correctness

2. Optimized Logical Plan:
Predicate pushdown (filter applied early),
Column pruning (only required columns scanned)

3. Physical Plan (Most Important):
FileScan,
PartitionFilters





In [0]:
    spark.sql("SELECT * FROM delta.`/Volumes/workspace/default/challenge/silver_events/` WHERE event_type='purchase'").explain(True)

#**************************************************

####Task 2 - Partition Large Tables

Partitioning physically splits data into folders based on column values.
Spark can skip entire folders when filtering.

In [0]:
spark.sql("""
CREATE TABLE silver_events_part
USING DELTA
PARTITIONED BY (event_date, event_type)
AS
SELECT * FROM delta.`/Volumes/workspace/default/challenge/silver_events/`
""")

#####Verify Partitions

In [0]:
%sql
SHOW PARTITIONS silver_events_part;


#**************************************************

####Task 3 - Apply ZORDER
- ZORDER reorders data inside files so related values are stored together.
- This improves data skipping for selective filters.

In [0]:
%sql
OPTIMIZE silver_events_part
ZORDER BY (user_id, product_id);


#**************************************************

####Task 4 - Benchmark Improvements
- Benchmarking increases query performance.
- When we run the below code before zorder and caching, the query time will be high.

#####To test query time

In [0]:
import time

start = time.time()
spark.sql(
    "SELECT * FROM silver_events_part WHERE user_id=12345"
).count()

print(f"Time: {time.time() - start:.2f}s")


#####Caching a frequently used table

In [0]:
# SERVERLESS COMPUTE DONT SUPPORT CACHING
cached = spark.table("silver_events_part").cache()
cached.count()  # Materialize cache


#**************************************************


%md
## **For me more such learning and insights in**
- ### [LinkedIn](https://www.linkedin.com/in/ilakkiyan-av/) 
- ### [Youtube](https://www.youtube.com/@ilakkiyanav) 